# Experiment 8: Model Evaluation & Prompt Engineering Benchmarks
This notebook evaluates model performance across prompt variations and computes Levenshtein Distance, Exact Match %, and Token F1.

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))
from src.corrector import GrammarCorrector
from src.evaluation import EvaluationModule
from src.error_analyzer import ErrorAnalyzerModule
from src.visualization import VisualizationModule

In [ ]:
# 1. Initialize Pipeline & Evaluator
corrector = GrammarCorrector(model_name_or_path=os.path.join('..', 'models', 'saved_models', 'grammar_corrector_model'))
evaluator = EvaluationModule()
analyzer = ErrorAnalyzerModule()

In [ ]:
# 2. Test Single Sentence Inference across Prompt Modes
sample = "He go to the laboratory yesterday for doing the experiment."
for mode in ['minimal', 'standard', 'rewrite', 'academic']:
    res = corrector.correct_sentence(sample, mode=mode)
    print(f"[{mode.upper()}]: {res}")

In [ ]:
# 3. Batch Evaluation on Benchmark Test Set
test_df = pd.read_csv(os.path.join('..', 'data', 'raw', 'lang8_errors.csv'))
inputs = test_df['error_sentence'].tolist()
refs = test_df['corrected_sentence'].tolist()
types = test_df['error_type'].tolist() if 'error_type' in test_df.columns else None

preds = corrector.batch_correct(inputs, mode='standard')
summary, results_df = evaluator.evaluate_batch(inputs, preds, refs, types)

print("Evaluation Summary Metrics:")
for k, v in summary.items():
    print(f"  {k}: {v}")

In [ ]:
# 4. Error Category Breakdown
grouped = analyzer.analyze_by_error_type(results_df)
print(grouped)